# Set Up

In [1]:
import requests
import pandas as pd
from datetime import datetime
import numpy as np

# Create Dataset

### API Call

In [2]:
API_KEY = "15a2f549a6e792af20fa6ece6aee9e52"

BASE_URL = "https://api.stlouisfed.org/fred/series/observations"
SERIES_URL = "https://api.stlouisfed.org/fred/series"

START_DATE = "1990-01-01"
END_DATE = "2026-06-30"

In [3]:
# List of FRED series
series = {
    "CPI": {
        "id": "CPIAUCSL",                # CPI All Urban Consumers
        "description": "Consumer Price Index"
    },
    "Unemployment Rate": {
        "id": "UNRATE",
        "description": "Civilian Unemployment Rate"
    },
    "Federal Funds Rate": {
        "id": "FEDFUNDS",
        "description": "Effective Federal Funds Rate"
    },
    "Yield Curve Spread": {
        "id": "T10Y2Y",
        "description": "10-Year Treasury Constant Maturity Minus 2-Year Treasury Constant Maturity"
    },
    "Industrial Production": {
        "id": "INDPRO",
        "description": "Industrial Production Index"
    },
    "Housing Starts": {
        "id": "HOUST",
        "description": "Housing Starts"
    },
    "Personal Consumption Expenditures": {
        "id": "PCE",
        "description": "Personal Consumption Expenditures"
    },
    "VIX": {
        "id": "VIXCLS",
        "description": "CBOE Volatility Index"
    },
    "Real GDP": {
        "id": "GDPC1",
        "description": "Real Gross Domestic Product"
    }
}

In [4]:
# Download Function
def download_series(series_id):

    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
        "observation_start": START_DATE,
        "observation_end": END_DATE
    }

    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()

    obs = r.json()["observations"]

    dates = []
    values = []

    for row in obs:

        value = row["value"]

        if value == ".":
            continue

        dates.append(row["date"])
        values.append(float(value))

    return {
        "Date": dates,
        "Value": values
    }

In [5]:
# Metadata Function
def get_metadata(series_id):

    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json"
    }

    r = requests.get(SERIES_URL, params=params)
    r.raise_for_status()

    s = r.json()["seriess"][0]

    return {
        "Frequency": s["frequency"],
        "Seasonal Adjustment": s["seasonal_adjustment"],
        "Start Date": s["observation_start"],
        "End Date": s["observation_end"]
    }

In [6]:
# Download Everything
summary = []

data = {}

for variable, info in series.items():

    print(f"Downloading {variable}...")

    data[variable] = download_series(info["id"])

    meta = get_metadata(info["id"])

    summary.append({
        "Variable": variable,
        "FRED Series": info["id"],
        "Frequency": meta["Frequency"],
        "Seasonally Adjusted": meta["Seasonal Adjustment"],
        "Start Date": meta["Start Date"],
        "End Date": meta["End Date"]
    })


# Individual Dictionaries
CPI = data["CPI"]

unemployment_rate = data["Unemployment Rate"]

federal_funds_rate = data["Federal Funds Rate"]

yield_curve_spread = data["Yield Curve Spread"]

industrial_production = data["Industrial Production"]

housing_starts = data["Housing Starts"]

personal_consumption_expenditures = data["Personal Consumption Expenditures"]

VIX = data["VIX"]

real_GDP = data["Real GDP"]

In [7]:
# Download Summary Table
summary_table = pd.DataFrame(summary)
summary_table

,Variable,FRED Series,Frequency,Seasonally Adjusted,Start Date,End Date
0,CPI,CPIAUCSL,Monthly,Seasonally Adjusted,1947-01-01,2026-07-01
1,Unemployment Rate,UNRATE,Monthly,Seasonally Adjusted,1948-01-01,2026-07-01
2,Federal Funds Rate,FEDFUNDS,Monthly,Not Seasonally Adjusted,1954-07-01,2026-07-01
3,Yield Curve Spread,T10Y2Y,Daily,Not Seasonally Adjusted,1976-06-01,2026-08-13
4,Industrial Production,INDPRO,Monthly,Seasonally Adjusted,1919-01-01,2026-06-01
5,Housing Starts,HOUST,Monthly,Seasonally Adjusted Annual Rate,1959-01-01,2026-06-01
6,Personal Consumption Expenditures,PCE,Monthly,Seasonally Adjusted Annual Rate,1959-01-01,2026-06-01
7,VIX,VIXCLS,"Daily, Close",Not Seasonally Adjusted,1990-01-02,2026-08-12
8,Real GDP,GDPC1,Quarterly,Seasonally Adjusted Annual Rate,1947-01-01,2026-04-01


In [24]:
markdown_table = summary_table.to_markdown(index=False)
print(markdown_table)

| Variable                          | FRED Series   | Frequency    | Seasonally Adjusted             | Start Date   | End Date   |
|:----------------------------------|:--------------|:-------------|:--------------------------------|:-------------|:-----------|
| CPI                               | CPIAUCSL      | Monthly      | Seasonally Adjusted             | 1947-01-01   | 2026-07-01 |
| Unemployment Rate                 | UNRATE        | Monthly      | Seasonally Adjusted             | 1948-01-01   | 2026-07-01 |
| Federal Funds Rate                | FEDFUNDS      | Monthly      | Not Seasonally Adjusted         | 1954-07-01   | 2026-07-01 |
| Yield Curve Spread                | T10Y2Y        | Daily        | Not Seasonally Adjusted         | 1976-06-01   | 2026-08-13 |
| Industrial Production             | INDPRO        | Monthly      | Seasonally Adjusted             | 1919-01-01   | 2026-06-01 |
| Housing Starts                    | HOUST         | Monthly      | Seasonally Adj

**Step 1 verification.** Confirm each series' frequency and seasonal-adjustment status match expectations, and flag anything at annual frequency (e.g. Real Median Household Income) so it can be excluded from the monthly panel rather than forced into it.

In [8]:
# Verify frequencies / seasonal adjustment, and flag any annual-frequency series
print(summary_table[["Variable", "FRED Series", "Frequency", "Seasonally Adjusted"]].to_string(index=False))

annual_series = summary_table.loc[
    summary_table["Frequency"].str.contains("Annual", case=False, na=False), "Variable"
].tolist()

if annual_series:
    print(f"\nAnnual-frequency series detected (will be excluded from the monthly panel): {annual_series}")
else:
    print("\nNo annual-frequency series in the current pull -- nothing to exclude on that basis.")


                         Variable FRED Series    Frequency             Seasonally Adjusted
                              CPI    CPIAUCSL      Monthly             Seasonally Adjusted
                Unemployment Rate      UNRATE      Monthly             Seasonally Adjusted
               Federal Funds Rate    FEDFUNDS      Monthly         Not Seasonally Adjusted
               Yield Curve Spread      T10Y2Y        Daily         Not Seasonally Adjusted
            Industrial Production      INDPRO      Monthly             Seasonally Adjusted
                   Housing Starts       HOUST      Monthly Seasonally Adjusted Annual Rate
Personal Consumption Expenditures         PCE      Monthly Seasonally Adjusted Annual Rate
                              VIX      VIXCLS Daily, Close         Not Seasonally Adjusted
                         Real GDP       GDPC1    Quarterly Seasonally Adjusted Annual Rate

No annual-frequency series in the current pull -- nothing to exclude on that basis.


### Cleaning Data

In [9]:
# Aggregate daily VIX to monthly two ways:
#   - VIX_MonthAvg: monthly average (preferred -- less sensitive to a single noisy day)
#   - VIX_MonthEnd: last trading day of the month (retained for comparison)
vix_df = pd.DataFrame(VIX)
vix_df["Date"] = pd.to_datetime(vix_df["Date"])
vix_df = vix_df.sort_values("Date")

vix_month_avg_df = (
    vix_df
    .groupby(vix_df["Date"].dt.to_period("M"))["Value"]
    .mean()
    .reset_index()
)
vix_month_avg_df["Date"] = vix_month_avg_df["Date"].dt.to_timestamp()

vix_month_end_df = (
    vix_df
    .groupby(vix_df["Date"].dt.to_period("M"), as_index=False)
    .last()
)
vix_month_end_df["Date"] = vix_month_end_df["Date"].dt.to_period("M").dt.to_timestamp()

In [10]:
# Same treatment for the 10Y-2Y Yield Curve Spread
yield_curve_spread_df = pd.DataFrame(yield_curve_spread)
yield_curve_spread_df["Date"] = pd.to_datetime(yield_curve_spread_df["Date"])
yield_curve_spread_df = yield_curve_spread_df.sort_values("Date")

yield_curve_spread_month_avg_df = (
    yield_curve_spread_df
    .groupby(yield_curve_spread_df["Date"].dt.to_period("M"))["Value"]
    .mean()
    .reset_index()
)
yield_curve_spread_month_avg_df["Date"] = (
    yield_curve_spread_month_avg_df["Date"].dt.to_timestamp()
)

yield_curve_spread_month_end_df = (
    yield_curve_spread_df
    .groupby(yield_curve_spread_df["Date"].dt.to_period("M"), as_index=False)
    .last()
)
yield_curve_spread_month_end_df["Date"] = (
    yield_curve_spread_month_end_df["Date"].dt.to_period("M").dt.to_timestamp()
)

In [11]:
# Real GDP is kept at its native quarterly frequency and is NOT interpolated to monthly,
# and is NOT merged into the monthly panel. Interpolating GDP to monthly would let each
# monthly point be informed by the *following* quarter's actual value -- look-ahead
# information that would not have been available at that date. Instead, GDP growth is
# exported as its own quarterly series, to be used later for validating whatever regimes
# get inferred from the monthly model (e.g. checking that a "contraction" regime lines up
# with quarters of negative GDP growth).
real_GDP_df = pd.DataFrame(real_GDP)
real_GDP_df["Date"] = pd.to_datetime(real_GDP_df["Date"])
real_GDP_df = real_GDP_df.sort_values("Date").reset_index(drop=True)

real_GDP_df["GDP_YoY_Growth"] = real_GDP_df["Value"].pct_change(4) * 100

gdp_quarterly_df = real_GDP_df.rename(columns={"Value": "Real_GDP"})[
    ["Date", "Real_GDP", "GDP_YoY_Growth"]
]

gdp_quarterly_df.head()

,Date,Real_GDP,GDP_YoY_Growth
0,1990-01-01,10047.386,NaN
1,1990-04-01,10083.855,NaN
2,1990-07-01,10090.569,NaN
3,1990-10-01,9998.704,NaN
4,1991-01-01,9951.916,-0.950197


In [12]:
# Recreate dictionaries for merging into the monthly panel.
# Month-average versions are the primary features; month-end versions are retained
# alongside them purely for comparison, per Step 1.
vix_month_avg = {
    "Date": vix_month_avg_df["Date"].dt.strftime("%Y-%m-%d").tolist(),
    "Value": vix_month_avg_df["Value"].tolist()
}
vix_month_end = {
    "Date": vix_month_end_df["Date"].dt.strftime("%Y-%m-%d").tolist(),
    "Value": vix_month_end_df["Value"].tolist()
}

yield_curve_spread_month_avg = {
    "Date": yield_curve_spread_month_avg_df["Date"].dt.strftime("%Y-%m-%d").tolist(),
    "Value": yield_curve_spread_month_avg_df["Value"].tolist()
}
yield_curve_spread_month_end = {
    "Date": yield_curve_spread_month_end_df["Date"].dt.strftime("%Y-%m-%d").tolist(),
    "Value": yield_curve_spread_month_end_df["Value"].tolist()
}

# Real GDP is intentionally NOT recreated here -- see gdp_quarterly_df above.

In [13]:
# List of dictionaries with desired column names.
# Real GDP is excluded (kept separate, see above). Any annual-frequency series flagged
# in the Step 1 verification cell are excluded here too, so the monthly panel only ever
# contains series that are genuinely monthly (or aggregated from daily).
variables = {
    "CPI": CPI,
    "Unemployment_Rate": unemployment_rate,
    "Federal_Funds_Rate": federal_funds_rate,
    "Yield_Curve_Spread_MonthAvg": yield_curve_spread_month_avg,
    "Yield_Curve_Spread_MonthEnd": yield_curve_spread_month_end,
    "Industrial_Production": industrial_production,
    "Housing_Starts": housing_starts,
    "Nominal_PCE": personal_consumption_expenditures,
    "VIX_MonthAvg": vix_month_avg,
    "VIX_MonthEnd": vix_month_end,
}

# Safety net: drop any variable whose underlying series was flagged as annual frequency
variables = {k: v for k, v in variables.items() if k not in annual_series}

# Convert each dictionary to dataframe and rename Value column
dfs = []

for name, dictionary in variables.items():

    df = pd.DataFrame(dictionary)

    # Convert dates
    df["Date"] = pd.to_datetime(df["Date"])

    # Rename value column to variable name
    df = df.rename(columns={"Value": name})

    dfs.append(df)


# Merge all variables on Date
monthly_macro_df = dfs[0]

for df in dfs[1:]:
    monthly_macro_df = monthly_macro_df.merge(
        df,
        on="Date",
        how="outer"
    )


# Sort chronologically
monthly_macro_df = (
    monthly_macro_df
    .sort_values("Date")
    .reset_index(drop=True)
)

In [14]:
# Limit to end 12/2025
monthly_macro_df = monthly_macro_df[monthly_macro_df["Date"] < pd.to_datetime('2026-01-01')]

In [15]:
# Fill Missing Gov't Statistics due to Shutdown in 10/2025 with the average of the previous and next month

# Set 'Date' column as index
monthly_macro_df = monthly_macro_df.set_index("Date")

monthly_macro_df.loc[pd.to_datetime("2025-10-01"), "CPI"] = (monthly_macro_df.loc[pd.to_datetime("2025-09-01"), "CPI"] + monthly_macro_df.loc[pd.to_datetime("2025-11-01"), "CPI"]) / 2
monthly_macro_df.loc[pd.to_datetime("2025-10-01"), "Unemployment_Rate"] = (monthly_macro_df.loc[pd.to_datetime("2025-09-01"), "Unemployment_Rate"] + monthly_macro_df.loc[pd.to_datetime("2025-11-01"), "Unemployment_Rate"]) / 2


In [16]:
monthly_macro_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 432 entries, 1990-01-01 to 2025-12-01
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CPI                          432 non-null    float64
 1   Unemployment_Rate            432 non-null    float64
 2   Federal_Funds_Rate           432 non-null    float64
 3   Yield_Curve_Spread_MonthAvg  432 non-null    float64
 4   Yield_Curve_Spread_MonthEnd  432 non-null    float64
 5   Industrial_Production        432 non-null    float64
 6   Housing_Starts               432 non-null    float64
 7   Nominal_PCE                  432 non-null    float64
 8   VIX_MonthAvg                 432 non-null    float64
 9   VIX_MonthEnd                 432 non-null    float64
dtypes: float64(10)
memory usage: 53.3 KB


##### Step 1 -> monthly_macro_df now holds only genuinely monthly (or monthly-aggregated) series in their raw levels/units -- Real GDP is excluded (kept quarterly, separately), and any annual-frequency series would have been excluded automatically above. Units still differ across columns (index levels, percentages, dollars, etc.)

In [17]:
# Quick check: confirm no missing values remain in the raw monthly panel
monthly_macro_df.isna().sum()

,0
CPI,0
Unemployment_Rate,0
Federal_Funds_Rate,0
Yield_Curve_Spread_MonthAvg,0
Yield_Curve_Spread_MonthEnd,0
Industrial_Production,0
Housing_Starts,0
Nominal_PCE,0
VIX_MonthAvg,0
VIX_MonthEnd,0


In [18]:
monthly_macro_df.head()

,CPI,Unemployment_Rate,Federal_Funds_Rate,Yield_Curve_Spread_MonthAvg,Yield_Curve_Spread_MonthEnd,Industrial_Production,Housing_Starts,Nominal_PCE,VIX_MonthAvg,VIX_MonthEnd
Date,,,,,,,,,,
1990-01-01,127.5,5.4,8.23,0.121429,0.15,61.7290,1551.0,3730.7,23.347273,25.36
1990-02-01,128.0,5.3,8.24,0.102632,0.08,62.2896,1437.0,3728.2,23.262632,21.99
1990-03-01,128.6,5.2,8.28,-0.038182,0.01,62.5999,1289.0,3754.9,20.062273,19.73
1990-04-01,128.9,5.4,8.26,0.061500,0.08,62.4359,1248.0,3770.0,21.403500,19.52
1990-05-01,129.1,5.4,8.18,0.115909,0.10,62.6258,1212.0,3775.8,18.097727,17.37


##### Step 2 -> Feature Engineering:

**Original (level) variables:** kept as-is. Alongside them, transformed versions are added wherever the level isn't economically stationary or wherever a transform makes the series more directly comparable/interpretable. Both versions are retained together in the same dataframe so raw vs. transformed can be compared during modeling.

**CPI, Industrial Production, Housing Starts:** -> add 12-month % change (YoY growth). Levels trend over time; YoY growth is the economically meaningful, roughly stationary read (inflation rate, industrial growth rate, housing-activity growth rate).

**Personal Consumption Expenditures:** -> add a deflated *Real_PCE* level (nominal PCE / CPI), plus its YoY growth. Nominal PCE growth would partly just re-encode CPI inflation; deflating first isolates real consumption growth.

**Unemployment Rate, Federal Funds Rate:** -> add the month-over-month change alongside the level. The level captures stance (restrictive/accommodative policy, high/low unemployment); the change captures direction (hiking/cutting, rising/falling unemployment -- in the spirit of the Sahm Rule).

**Yield Curve Spread:** -> no additional transform. It's already mean-reverting around zero (a spread, not a level with a trend), so both the month-average and month-end versions are used as-is.

**VIX:** -> add a log transform (of both the month-average and month-end versions). VIX is strongly right-skewed; log(VIX) is much closer to normal.

**Real GDP:** -> not part of this feature set at all (kept quarterly, separate -- see Step 1).

In [19]:
# Start from the raw monthly panel and add transformed columns alongside the originals
master_df = monthly_macro_df.copy()

# --- Real PCE: deflate nominal PCE by CPI to isolate real consumption ---
master_df["Real_PCE"] = (
    master_df["Nominal_PCE"] / (master_df["CPI"] / 100)
)

# --- YoY growth rates (12-month % change) ---
master_df["CPI_YoY"] = master_df["CPI"].pct_change(12) * 100
master_df["Industrial_Production_YoY"] = master_df["Industrial_Production"].pct_change(12) * 100
master_df["Housing_Starts_YoY"] = master_df["Housing_Starts"].pct_change(12) * 100
master_df["Real_PCE_YoY"] = master_df["Real_PCE"].pct_change(12) * 100

# --- Month-over-month changes ---
master_df["Unemployment_Rate_MoM_Change"] = master_df["Unemployment_Rate"].diff()
master_df["Federal_Funds_Rate_MoM_Change"] = master_df["Federal_Funds_Rate"].diff()

# --- Log transform of VIX (both variants) ---
master_df["Log_VIX_MonthAvg"] = np.log(master_df["VIX_MonthAvg"])
master_df["Log_VIX_MonthEnd"] = np.log(master_df["VIX_MonthEnd"])

# Note: rows in the first 12 months will have NaNs in the YoY columns (insufficient
# history for a 12-month lookback), and the first row will have a NaN in the two
# _Change columns. These are NOT dropped here -- raw levels are still valid for that
# period, and it's cleaner to let the modeling notebook decide how to handle the
# warm-up window depending on which feature subset it actually uses.
master_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 432 entries, 1990-01-01 to 2025-12-01
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   CPI                            432 non-null    float64
 1   Unemployment_Rate              432 non-null    float64
 2   Federal_Funds_Rate             432 non-null    float64
 3   Yield_Curve_Spread_MonthAvg    432 non-null    float64
 4   Yield_Curve_Spread_MonthEnd    432 non-null    float64
 5   Industrial_Production          432 non-null    float64
 6   Housing_Starts                 432 non-null    float64
 7   Nominal_PCE                    432 non-null    float64
 8   VIX_MonthAvg                   432 non-null    float64
 9   VIX_MonthEnd                   432 non-null    float64
 10  Real_PCE                       432 non-null    float64
 11  CPI_YoY                        420 non-null    float64
 12  Industrial_Production_YoY      

##### Step 3. Feature Review

Check correlations/redundancy across the full raw + transformed set before deciding what actually goes into the regime model.

In [20]:
# Correlation matrix over complete cases (drops the ~12-month YoY warm-up window)
corr_matrix = master_df.dropna().corr()
corr_matrix.round(2)

,CPI,Unemployment_Rate,Federal_Funds_Rate,Yield_Curve_Spread_MonthAvg,Yield_Curve_Spread_MonthEnd,Industrial_Production,Housing_Starts,Nominal_PCE,VIX_MonthAvg,VIX_MonthEnd,Real_PCE,CPI_YoY,Industrial_Production_YoY,Housing_Starts_YoY,Real_PCE_YoY,Unemployment_Rate_MoM_Change,Federal_Funds_Rate_MoM_Change,Log_VIX_MonthAvg,Log_VIX_MonthEnd
CPI,1.00,-0.22,-0.36,-0.26,-0.26,0.78,-0.16,1.00,0.00,0.01,0.99,0.12,-0.26,-0.02,-0.07,-0.00,0.13,0.01,0.01
Unemployment_Rate,-0.22,1.00,-0.48,0.72,0.71,-0.33,-0.60,-0.27,0.19,0.18,-0.27,-0.34,-0.24,0.05,-0.40,0.16,-0.14,0.19,0.17
Federal_Funds_Rate,-0.36,-0.48,1.00,-0.64,-0.63,-0.40,0.40,-0.31,-0.13,-0.15,-0.40,0.28,0.29,-0.25,0.14,0.01,-0.02,-0.11,-0.13
Yield_Curve_Spread_MonthAvg,-0.26,0.72,-0.64,1.00,1.00,-0.23,-0.44,-0.30,0.11,0.10,-0.26,-0.32,-0.12,0.17,-0.10,-0.01,-0.23,0.10,0.09
Yield_Curve_Spread_MonthEnd,-0.26,0.71,-0.63,1.00,1.00,-0.23,-0.44,-0.30,0.12,0.11,-0.26,-0.32,-0.13,0.16,-0.11,-0.01,-0.25,0.11,0.11
Industrial_Production,0.78,-0.33,-0.40,-0.23,-0.23,1.00,0.02,0.76,0.04,0.06,0.85,-0.02,-0.13,-0.03,0.04,-0.04,0.12,0.05,0.07
Housing_Starts,-0.16,-0.60,0.40,-0.44,-0.44,0.02,1.00,-0.11,-0.17,-0.15,-0.09,0.28,0.27,0.20,0.41,-0.08,0.19,-0.14,-0.13
Nominal_PCE,1.00,-0.27,-0.31,-0.30,-0.30,0.76,-0.11,1.00,-0.02,-0.01,0.98,0.15,-0.24,-0.02,-0.03,-0.02,0.13,-0.01,-0.01
VIX_MonthAvg,0.00,0.19,-0.13,0.11,0.12,0.04,-0.17,-0.02,1.00,0.93,-0.00,-0.06,-0.37,-0.29,-0.31,0.19,-0.33,0.97,0.90
VIX_MonthEnd,0.01,0.18,-0.15,0.10,0.11,0.06,-0.15,-0.01,0.93,1.00,0.01,-0.06,-0.33,-0.22,-0.27,0.13,-0.29,0.91,0.97


In [21]:
# Flag highly correlated / likely-redundant pairs (|correlation| above threshold)
THRESHOLD = 0.8

redundant_pairs = (
    corr_matrix
    .where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .rename("correlation")
    .reset_index()
    .rename(columns={"level_0": "Feature_1", "level_1": "Feature_2"})
)
redundant_pairs = redundant_pairs[redundant_pairs["correlation"].abs() >= THRESHOLD]
redundant_pairs = redundant_pairs.sort_values("correlation", key=abs, ascending=False)

redundant_pairs

,Feature_1,Feature_2,correlation
6,CPI,Nominal_PCE,0.996223
51,Yield_Curve_Spread_MonthAvg,Yield_Curve_Spread_MonthEnd,0.996037
9,CPI,Real_PCE,0.985988
107,Nominal_PCE,Real_PCE,0.982622
134,VIX_MonthEnd,Log_VIX_MonthEnd,0.971156
124,VIX_MonthAvg,Log_VIX_MonthAvg,0.966473
170,Log_VIX_MonthAvg,Log_VIX_MonthEnd,0.937762
116,VIX_MonthAvg,VIX_MonthEnd,0.931620
133,VIX_MonthEnd,Log_VIX_MonthAvg,0.912292
125,VIX_MonthAvg,Log_VIX_MonthEnd,0.901620


In [22]:
candidate_features = [
    "CPI_YoY",
    "Industrial_Production_YoY",
    "Housing_Starts_YoY",
    "Real_PCE_YoY",
    "Unemployment_Rate",
    "Unemployment_Rate_MoM_Change",
    "Federal_Funds_Rate",
    "Federal_Funds_Rate_MoM_Change",
    "Yield_Curve_Spread_MonthAvg",
    "Log_VIX_MonthAvg",
]

candidate_features

['CPI_YoY',
 'Industrial_Production_YoY',
 'Housing_Starts_YoY',
 'Real_PCE_YoY',
 'Unemployment_Rate',
 'Unemployment_Rate_MoM_Change',
 'Federal_Funds_Rate',
 'Federal_Funds_Rate_MoM_Change',
 'Yield_Curve_Spread_MonthAvg',
 'Log_VIX_MonthAvg']

##### Step 4. Export the Dataset


In [23]:
master_df.to_csv("master_monthly_dataset.csv")
gdp_quarterly_df.to_csv("gdp_quarterly_growth.csv", index=False)

print("Master monthly dataset:", master_df.shape)
print("Quarterly GDP validation series:", gdp_quarterly_df.shape)

Master monthly dataset: (432, 19)
Quarterly GDP validation series: (146, 3)
